[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/11_Chunking_and_Splitting.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 分块与切分（Chunking and Splitting）—— 全面指南

## 概览

本 notebook 全面走读 Semantica 的 split 模块，演示全部分块策略与方法。你将学会 15+ 种切分方法，覆盖标准、语义和知识图谱感知三类途径。

**文档**：[API 参考](https://semantica.readthedocs.io/reference/split/)

### 先看主线

为什么分块重要？流水线里它排在规范化之后、抽取之前：LLM 上下文有限、检索需要合适粒度、抽取需要完整上下文——**块是后面一切工序的工作单元**。切得好不好，直接影响 05/06 篇的抽取质量和 12 篇的嵌入质量。

本篇共 13 步（上游编号从第 11 步直接跳到第 13 步，中间没有第 12 步——不是你漏看了），按"四字纲"分四段：

**切得动（第 1–3 步）—— 物理切法**

按字符、句子、段落、token 数机械地切。统一入口是 `TextSplitter`。

**切得准（第 4–6 步）—— 语义与图谱切法**

第 4 步按主题边界切（语义分块）；第 5、6 步为图谱保实体、保三元组（GraphRAG 切法）。

**切得巧（第 7–10 步）—— 特殊场景**

结构化文档、多粒度层次、滑动窗口、表格，各有专用工具。

**切得清（第 11–14 步）—— 保障与选型**

溯源追踪、方法跑分对比、最佳实践——没有万能切法，**选型才是本篇真正的主题**。

### 学习目标

完成本 notebook 后，你将能够：

- 使用多种方法的 `TextSplitter`
- 应用标准切分方法（recursive、token、sentence、paragraph）
- 使用语义分块保证主题连贯
- 应用 KG 感知分块（实体感知、关系感知、图感知）
- 使用专用分块器（结构化、滑动窗口、表格、层次）
- 用 `ProvenanceTracker` 追踪溯源
- 为你的场景选对方法

### 你将学到什么

| 组件 | 用途 | 适用场景 |
|-----------|---------|-------------|
| `TextSplitter` | 统一切分器 | 所有分块需求 |
| `SemanticChunker` | 语义边界 | 按主题分块 |
| `EntityAwareChunker` | 保护实体 | GraphRAG 工作流 |
| `RelationAwareChunker` | 保护三元组 | KG 构建 |
| `StructuralChunker` | 文档结构 | 格式化文档 |
| `HierarchicalChunker` | 多级块 | 大型文档 |

---

## 安装

从 PyPI 安装 Semantica：

```bash
pip install semantica
# 或安装全部可选依赖：
pip install semantica[all]
```

---

In [ ]:
!pip install -q semantica


## 第 1 步：用 TextSplitter 做基础分块

主线从"切得动"开始：先认识统一入口 `TextSplitter`——一种接口通吃所有分块方法。

### 什么是 TextSplitter？

`TextSplitter` 是统一接口，支持 15+ 种分块方法：
- **标准**：recursive、token、sentence、paragraph、character、word
- **语义**：semantic_transformer、llm、huggingface、nltk
- **KG/本体**：entity_aware、relation_aware、graph_based、ontology_aware
- **高级**：hierarchical、structural、sliding_window、table

In [ ]:
from semantica.split import TextSplitter

# Sample long text
text = """
Apple Inc. is a technology company founded by Steve Jobs, Steve Wozniak, and Ronald Wayne 
in Cupertino, California on April 1, 1976. The company's current CEO is Tim Cook, who took 
over from Steve Jobs in August 2011. Apple is headquartered at One Apple Park Way in Cupertino.

Apple develops and sells consumer electronics, computer software, and online services. The company's 
hardware products include the iPhone smartphone, the iPad tablet computer, the Mac personal computer, 
the iPod portable media player, the Apple Watch smartwatch, the Apple TV digital media player, and the 
HomePod smart speaker.

Apple's software includes the macOS and iOS operating systems, the iTunes media player, the Safari web 
browser, and the iLife and iWork creativity and productivity suites. Its online services include the 
iTunes Store, the iOS App Store and Mac App Store, Apple Music, and iCloud.
"""

# Basic recursive splitting
splitter = TextSplitter(
    method="recursive",
    chunk_size=200,
    chunk_overlap=50
)

chunks = splitter.split(text)

print(f"Split into {len(chunks)} chunks using recursive method\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    print(f"\nChunk {i}:")
    print(f"  Length: {len(chunk.text)} characters")
    print(f"  Start: {chunk.start_index}, End: {chunk.end_index}")
    print(f"  Text: {chunk.text[:100]}...")

print("\n" + "=" * 80)

## 第 2 步：标准切分方法对比

物理切法不止一种。同一段文本交给不同方法，块数、块形各不相同：

### 方法对比

| 方法 | 最适合 | 速度 | 准确度 |
|--------|----------|-------|----------|
| **recursive** | 通用文本 | 快 | 好 |
| **sentence** | 连贯块 | 中 | 很好 |
| **token** | LLM 上下文 | 中 | 极好 |
| **paragraph** | 自然断点 | 快 | 好 |

In [ ]:
# Compare different methods
methods = ["recursive", "sentence", "paragraph"]

print("Comparing Standard Splitting Methods:\n")
print("=" * 80)

for method in methods:
    splitter = TextSplitter(
        method=method,
        chunk_size=200,
        chunk_overlap=50
    )
    
    chunks = splitter.split(text)
    
    print(f"\nMethod: {method.upper()}")
    print("-" * 40)
    print(f"  Chunks created: {len(chunks)}")
    print(f"  Avg chunk size: {sum(len(c.text) for c in chunks) / len(chunks):.0f} chars")
    print(f"  First chunk: {chunks[0].text[:80]}...")

print("\n" + "=" * 80)

## 第 3 步：基于 token 的切分

物理切法里最讲究的一种：按 token 数切。凡是喂给 LLM 的场景都绕不开它。

### 为什么要按 token？

- **LLM 上下文窗口**：GPT-4 有 8K/32K token 上限
- **精确计数**：字符数 ≠ token 数
- **成本优化**：API 按 token 计费

In [ ]:
from semantica.split import split_by_tokens

# Token-based splitting
chunks = split_by_tokens(
    text,
    chunk_size=100,  # 100 tokens
    chunk_overlap=20,
    tokenizer="tiktoken",
    model="gpt-4"
)

print("Token-Based Splitting Results:\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    token_count = chunk.metadata.get('token_count', 'N/A')
    print(f"\nChunk {i}:")
    print(f"  Tokens: {token_count}")
    print(f"  Characters: {len(chunk.text)}")
    print(f"  Ratio: {len(chunk.text)/token_count if token_count != 'N/A' else 'N/A':.2f} chars/token")

print("\n" + "=" * 80)

## 第 4 步：语义分块

"切得动"解决，开始"切得准"。物理切法不认识主题边界——硬切可能把一个话题拦腰斩断。语义分块用嵌入找主题边界，切出来的块各说各的事。

### 工作原理

1. 把文本切成句子
2. 为每句生成嵌入（embedding）
3. 计算相邻句子的相似度
4. 相似度跌破阈值处设边界

In [ ]:
from semantica.split import SemanticChunker

# Semantic chunking
semantic_chunker = SemanticChunker(
    chunk_size=200,
    chunk_overlap=50,
    embedding_model="all-MiniLM-L6-v2",
    similarity_threshold=0.7
)

chunks = semantic_chunker.chunk(text)

print("Semantic Chunking Results:\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    coherence = chunk.metadata.get('coherence_score', 'N/A')
    print(f"\nChunk {i}:")
    print(f"  Length: {len(chunk.text)} chars")
    print(f"  Coherence: {coherence}")
    print(f"  Text: {chunk.text[:100]}...")

print("\n" + "=" * 80)

## 第 5 步：面向 GraphRAG 的实体感知分块

"Steve Jobs" 被切成两个块，05 篇的 NER 就抓瞎了。实体感知分块保证实体不被切断——GraphRAG 工作流的关键前置。

### 为什么要实体感知？

- **保护实体**：不把 "Steve Jobs" 拆到两个块
- **抽取更准**：完整实体提升 NER 准确率
- **GraphRAG**：知识图谱构建的必要保障

In [ ]:
from semantica.split import EntityAwareChunker

# Entity-aware chunking
entity_chunker = EntityAwareChunker(
    chunk_size=200,
    chunk_overlap=50,
    ner_method="ml",  # "ml" (spaCy), "pattern", or "llm"
    preserve_entities=True
)

chunks = entity_chunker.chunk(text)

print("Entity-Aware Chunking Results:\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    entities = chunk.metadata.get('entities', [])
    print(f"\nChunk {i}:")
    print(f"  Length: {len(chunk.text)} chars")
    print(f"  Entities: {len(entities)}")
    
    if entities:
        # Handle both Entity objects and dicts
        entity_texts = [e.get('text', e.get('entity', '')) if isinstance(e, dict) else str(e) for e in entities[:3]]
        print(f"  Sample entities: {entity_texts}")

print("\n" + "=" * 80)

## 第 6 步：关系感知分块

实体保住了，还要保关系。关系感知分块让三元组 (subject, predicate, object) 完整待在同一个块里——06 篇的关系抽取依赖完整上下文。

### 为什么要关系感知？

- **保护三元组**：让 (主语, 谓语, 宾语) 不拆散
- **KG 构建**：更适合构建知识图谱
- **上下文**：关系抽取需要完整上下文

In [ ]:
from semantica.split import RelationAwareChunker

# Relation-aware chunking
relation_chunker = RelationAwareChunker(
    chunk_size=200,
    chunk_overlap=50,
    preserve_triplets=True
)

chunks = relation_chunker.chunk(text)

print("Relation-Aware Chunking Results:\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    triplets = chunk.metadata.get('triplets', [])
    relationships = chunk.metadata.get('relationships', [])
    
    print(f"\nChunk {i}:")
    print(f"  Length: {len(chunk.text)} chars")
    print(f"  Triplets: {len(triplets)}")
    print(f"  Relationships: {len(relationships)}")

print("\n" + "=" * 80)

## 第 7 步：结构化分块

进入"切得巧"段。带格式的文档（Markdown、HTML）自带结构：标题、段落、列表就是天然的边界。

### 什么时候用？

- **格式化文档**：Markdown、HTML、结构化文本
- **保留层级**：让小节完整待在一起
- **更好的上下文**：标题自带上下文

In [ ]:
from semantica.split import StructuralChunker

# Markdown text with structure
markdown_text = """
# Apple Inc.

## History

Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in 1976.

## Products

### Hardware
- iPhone
- iPad
- Mac

### Software
- macOS
- iOS
- Safari
"""

# Structural chunking
structural_chunker = StructuralChunker(
    respect_headings=True,
    respect_paragraphs=True,
    respect_lists=True,
    max_chunk_size=500
)

chunks = structural_chunker.chunk(markdown_text)

print("Structural Chunking Results:\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    section = chunk.metadata.get('section_title', 'N/A')
    level = chunk.metadata.get('heading_level', 'N/A')
    
    print(f"\nChunk {i}:")
    print(f"  Section: {section}")
    print(f"  Level: {level}")
    print(f"  Text: {chunk.text[:80]}...")

print("\n" + "=" * 80)

## 第 8 步：层次分块

大文档一种粒度不够用：粗粒度给概览、细粒度给检索。层次分块切出多级块，父子相连。

### 好处

- **多粒度**：文档 → 小节 → 段落
- **更好导航**：父子关系可遍历
- **灵活检索**：按不同层级查询

In [ ]:
from semantica.split import HierarchicalChunker

# Hierarchical chunking
hierarchical_chunker = HierarchicalChunker(
    chunk_sizes=[400, 200, 100],  # 3 levels
    chunk_overlaps=[80, 40, 20],
    create_parent_chunks=True
)

chunks = hierarchical_chunker.chunk(text)

print("Hierarchical Chunking Results:\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    level = chunk.metadata.get('level', 'N/A')
    parent_id = chunk.metadata.get('parent_id', None)
    child_ids = chunk.metadata.get('child_ids', [])
    
    print(f"\nChunk {i}:")
    print(f"  Level: {level}")
    print(f"  Length: {len(chunk.text)} chars")
    print(f"  Parent: {parent_id if parent_id else 'None (root)'}")
    print(f"  Children: {len(child_ids)}")

print("\n" + "=" * 80)

## 第 9 步：滑动窗口分块

固定大小、固定步长、块块重叠——最简单也最保险的"地毯式搜索"切法。

### 适用场景

- **密集检索**：确保信息不被边界漏掉
- **固定上下文**：块大小一致
- **精确控制重叠**：重叠量可控

In [ ]:
from semantica.split import SlidingWindowChunker

# Sliding window chunking
sliding_chunker = SlidingWindowChunker(
    chunk_size=150,
    overlap=50
)

chunks = sliding_chunker.chunk(text)

print("Sliding Window Chunking Results:\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    # Calculate overlap manually
    overlap = 0
    if i > 1:
        prev_chunk = chunks[i-2]
        overlap = max(0, prev_chunk.end_index - chunk.start_index)
    
    print(f"\nWindow {i}:")
    print(f"  Position: {chunk.start_index}-{chunk.end_index}")
    print(f"  Length: {len(chunk.text)} chars")
    print(f"  Overlap with previous: {overlap} chars")

print("\n" + "=" * 80)

## 第 10 步：表格分块

表格按字符切就废了。表格分块按行切、保留表头——大表拆小表，小表仍是完整的表。

### 特性

- **保留表头**：每个块都带列名
- **按行切分**：按行不按字符
- **包含上下文**：带上表格前后的文字

In [ ]:
from semantica.split import TableChunker

# Text with table
text_with_table = """
Apple's product lineup includes:

| Product | Category | Release Year |
|---------|----------|-------------|
| iPhone | Smartphone | 2007 |
| iPad | Tablet | 2010 |
| Mac | Computer | 1984 |
| Apple Watch | Wearable | 2015 |
| AirPods | Audio | 2016 |

These products have revolutionized their respective categories.
"""

# Table chunking
table_chunker = TableChunker(
    preserve_headers=True,
    max_rows_per_chunk=3,
    include_context=True,
    table_format="markdown"
)

chunks = table_chunker.chunk(text_with_table)

print("Table Chunking Results:\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    is_table = chunk.metadata.get('is_table', False)
    
    print(f"\nChunk {i}:")
    print(f"  Type: {'Table' if is_table else 'Text'}")
    
    if is_table:
        rows = chunk.metadata.get('row_count', 'N/A')
        cols = chunk.metadata.get('column_count', 'N/A')
        print(f"  Rows: {rows}, Columns: {cols}")
    
    print(f"  Content: {chunk.text[:100]}...")

print("\n" + "=" * 80)

## 第 11 步：溯源追踪

切法讲完，进入"切得清"段：块从哪来，要能查回去——这是 Semantica 溯源纪律在 split 模块的体现。

### 为什么要溯源？

- **数据血缘**：知道每块来自哪里
- **调试**：问题顺着块追回原文
- **合规**：部分场景的硬性要求

In [ ]:
import sys
import os
import importlib

# 1. Ensure local package is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 2. Force unload modules to ensure clean reload
modules_to_unload = [
    'semantica.split.semantic_chunker', 
    'semantica.split.splitter', 
    'semantica.split.provenance_tracker',
    'semantica.split'
]
for module in modules_to_unload:
    if module in sys.modules:
        del sys.modules[module]

# 3. Import fresh modules
import semantica.split.semantic_chunker
import semantica.split.splitter
import semantica.split.provenance_tracker
from semantica.split import ProvenanceTracker, TextSplitter

# 4. Verify Chunk class has id field
from semantica.split.semantic_chunker import Chunk
print(f"Chunk class fields: {Chunk.__annotations__}")
if 'id' not in Chunk.__annotations__:
    print("WARNING: Chunk class still missing 'id' field. Kernel restart required.")

# Create chunks
splitter = TextSplitter(method="recursive", chunk_size=200, chunk_overlap=50)
chunks = splitter.split(text)

# Track provenance
tracker = ProvenanceTracker()

for chunk in chunks:
    tracker.track_chunk(
        chunk=chunk,
        source_document="apple_doc_001",
        source_path="data/apple.txt",
        timestamp="2024-01-01T00:00:00Z",
        method="recursive"
    )

print("Provenance Tracking Results:\n")
print("=" * 80)

# Get lineage for first chunk
if chunks:
    # Get provenance info using the chunk's ID
    chunk_id = getattr(chunks[0], 'id', None)
    print(f"Chunk ID: {chunk_id}")
    
    if chunk_id:
        prov_info = tracker.get_provenance(chunk_id)
        
        if prov_info:
            print(f"\nLineage for Chunk 1:")
            print(f"  Source Document: {prov_info.source_document}")
            print(f"  File Path: {prov_info.source_path}")
            print(f"  Method: {prov_info.metadata.get('method')}")
            print(f"  Timestamp: {prov_info.timestamp}")
    else:
        print("Error: Chunk ID not found. The Chunk class definition might still be cached.")
        print("Please click 'Kernel' -> 'Restart Kernel' in the menu and run all cells again.")

print("\n" + "=" * 80)

## 第 13 步：方法对比

（上游编号从第 11 步直接跳到这里，没有第 12 步。）切法这么多，跑个分眼见为实：

### 对比维度

- **块数**：产出多少块
- **平均大小**：块的平均尺寸
- **处理耗时**：切分速度

In [ ]:
import time

# Methods to compare
methods_to_compare = [
    ("recursive", {}),
    ("sentence", {}),
    ("paragraph", {}),
    ("token", {"tokenizer": "tiktoken"}),
]

print("Method Comparison:\n")
print("=" * 80)
print(f"{'Method':<15} {'Chunks':<10} {'Avg Size':<12} {'Time (ms)':<12}")
print("-" * 80)

for method, kwargs in methods_to_compare:
    try:
        start_time = time.time()
        
        splitter = TextSplitter(
            method=method,
            chunk_size=200,
            chunk_overlap=50,
            **kwargs
        )
        
        chunks = splitter.split(text)
        
        elapsed = (time.time() - start_time) * 1000
        avg_size = sum(len(c.text) for c in chunks) / len(chunks) if chunks else 0
        
        print(f"{method:<15} {len(chunks):<10} {avg_size:<12.0f} {elapsed:<12.2f}")
        
    except Exception as e:
        print(f"{method:<15} Error: {str(e)[:40]}")

print("=" * 80)

## 第 14 步：最佳实践

### 选对方法

1. **通用文档**：`recursive`，快而简单
2. **LLM 应用**：`token`，尊重上下文窗口
3. **语义检索**：`semantic_transformer`，主题连贯
4. **GraphRAG**：`entity_aware` 或 `relation_aware`
5. **结构化文档**：`structural`
6. **大型文档**：`hierarchical`，多级访问

### 块大小参考

| 用途 | 推荐大小 | 重叠 |
|----------|------------------|----------|
| 语义检索 | 512-1024 字符 | 20% |
| LLM 上下文 | 2000-4000 字符 | 10-20% |
| 实体抽取 | 500-1500 字符 | 15-25% |
| 问答 | 1000-2000 字符 | 20% |

### 重叠比例建议

- **10-15%**：处理快、冗余少
- **20-25%**：均衡（推荐）
- **30-40%**：最大程度保留上下文

## 小结

### 回顾主线

四字纲收拢：**切得动**（物理切法：字符、句子、token）解决能不能切；**切得准**（语义边界、实体与三元组保护）解决切在哪；**切得巧**（结构、层次、滑窗、表格）解决特殊场景；**切得清**（溯源、跑分、选型）解决怎么选。块是抽取与嵌入的工作单元——原料备好了，12 篇就该给它们生成嵌入。

### 你学到了什么

在本 notebook 中，你学会了：

- 使用 `TextSplitter` 的多种方法
- 应用标准切分（recursive、token、sentence、paragraph）
- 使用语义分块保证主题连贯
- 应用 KG 感知分块（实体感知、关系感知）
- 使用专用分块器（结构化、层次、滑动窗口、表格）
- 追踪溯源
- 为你的场景选对方法

### 核心要点

1. **方法选型很重要**：不同需求不同方法
2. **块大小是关键**：上下文与处理开销的平衡
3. **重叠有帮助**：20% 是不错的默认值
4. **追踪溯源**：调试与合规都需要
5. **GraphRAG 用 KG 感知切法**：建图谱就选实体/关系感知

### 下一步

**下一篇**：[12_Embedding_Generation.ipynb](./12_Embedding_Generation.ipynb)  
学习如何为你的块生成嵌入！

**延伸阅读**：
- [Split 模块 API 参考](https://semantica.readthedocs.io/reference/split/)
- [高级分块策略](../advanced/11_Text_Chunking_Strategies.ipynb)
- [GraphRAG 流水线](../use_cases/advanced_rag/01_GraphRAG_Complete.ipynb)

---

**有疑问或问题？** 欢迎访问我们的 [GitHub 仓库](https://github.com/semantica-agi/semantica)或[文档](https://semantica.readthedocs.io)。